# FIN510 Coursework 2 Option B | Market prediction

Would you change the forecast you give to an investment team? Compare two models for next month's market excess return with the historical mean. Keep the market, forecast horizon and dates the same for all three methods so that a difference in forecast error has a clear meaning.

Choose one market, the US (`usa`) or the UK (`gbr`). Decide what size and consistency of improvement would matter before you look at the test results. In Colab, upload this notebook and run the cells in order. The first code cell downloads the pinned course data when the CSV is absent from the working directory.

## What is known when you make a forecast?

At the end of April 2010, the target is the market excess return for **May 2010**. The benchmark uses market excess returns observed through April. The regression also trains only on outcomes through April. Its four predictors for May come from **March**, one full calendar month before the forecast origin. The delay is a conservative assumption for the lagged inputs. The course extract does not record vintage or release times, so discuss what data an actual forecasting team would have had at the time.

In [ ]:
import hashlib
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

market = "usa"  # Use "gbr" for the United Kingdom.
market_names = {"usa": "United States", "gbr": "United Kingdom"}
if market not in market_names:
    raise ValueError("Choose usa or gbr for the sample in this starter.")

data_url = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "0816bab2cb8e7468f559ca69494ae2776139f1c6/labs/jkp_master_global_monthly.csv"
)
expected_sha256 = "6a4eb3034551694041a149625777a222d3e3ac0a4e442b2b85f759347b28d24d"
local_data = Path("jkp_master_global_monthly.csv")
if local_data.is_file():
    payload = local_data.read_bytes()
else:
    with urlopen(data_url, timeout=30) as response:
        payload = response.read()
if hashlib.sha256(payload).hexdigest() != expected_sha256:
    raise ValueError("Use the pinned course CSV linked below.")

source_columns = ["MKT", "SMB", "HML", "MOM"]
source = pd.read_csv(BytesIO(payload), usecols=["country", "date", *source_columns])
selected = source.loc[source["country"].eq(market)].copy()
selected["Month"] = pd.to_datetime(selected["date"]).dt.to_period("M")
monthly = selected.set_index("Month")[source_columns].sort_index().mul(100)
if monthly.empty or not monthly.index.is_unique:
    raise ValueError("Check that this market has one return row per calendar month.")

calendar = pd.period_range(monthly.index[0], monthly.index[-1], freq="M", name="Month")
monthly = monthly.reindex(calendar)
print(f"{market_names[market]} | {monthly.index[0]} to {monthly.index[-1]} in the extract")

## Put the outcome and the inputs on the same row

In the [pinned course extract](https://github.com/quinfer/fin510-colab-notebooks/blob/0816bab2cb8e7468f559ca69494ae2776139f1c6/labs/jkp_master_global_monthly.csv), `MKT` is the monthly market excess return in USD, including for the UK. The course file uses decimals. Multiplying by 100 expresses returns in per cent. `SMB`, `HML` and `MOM` label the size, value and momentum series in this extract. These three lagged factor returns and the lagged market return give both models the same candidate inputs. Explain why these inputs are plausible before judging the results. Cite and describe your data in the report. [JKP data definitions](https://jkpfactors.com/data)

Each row below represents a **target month**. The two-month shift puts March's inputs on May's row. We first insert any missing calendar months, so missing March data remain missing on May's row. Check the first rows and describe which observation becomes available next at each forecast origin.

In [ ]:
first_target = pd.Period("1990-01", freq="M")
first_validation = pd.Period("2007-01", freq="M")
first_test = pd.Period("2010-01", freq="M")
last_test = pd.Period("2023-12", freq="M")
robustness_break = pd.Period("2017-01", freq="M")
feature_columns = [f"{column} lag" for column in source_columns]
target_column = "Actual MKT"


def align_months(returns):
    lagged_inputs = returns[source_columns].shift(2)
    lagged_inputs.columns = feature_columns
    lagged_inputs[target_column] = returns["MKT"]
    return lagged_inputs.loc[first_target:last_test]


design = align_months(monthly)
test_calendar = design.loc[first_test:last_test].index
timing = pd.DataFrame({
    "Target month": test_calendar[:3].astype(str),
    "Forecast issued at end of": (test_calendar[:3] - 1).astype(str),
    "Month of predictor returns": (test_calendar[:3] - 2).astype(str),
})
display(timing)

## Choose the ridge penalty with earlier forecasts

Both models include an intercept. Ridge adds a penalty for large slopes. We standardise the four inputs using the mean and scale of the training rows at each forecast origin, so ridge penalises slopes measured on a comparable scale. The pipeline fits a new scaler and model each time. The observed target stays in return units.

Use the same one-month forecasting procedure to compare ridge penalties from January 2007 through December 2009. Issue each validation forecast with a growing training sample and score it once the outcome is known. Choose the penalty with the lowest validation mean squared forecast error, then keep that penalty fixed in the later test. The fitted coefficients and scaler will still update each month.

In [ ]:
def fit_forecast(data, target_month, estimator):
    origin = target_month - 1
    training = data.loc[:origin].dropna()
    if len(training) < 24:
        raise ValueError(f"Too few complete training rows at {origin}.")
    fitted = make_pipeline(StandardScaler(), estimator)
    fitted.fit(training[feature_columns], training[target_column])
    current_inputs = data.loc[[target_month], feature_columns]
    prediction = float(fitted.predict(current_inputs)[0])
    return prediction, len(training)


penalties = [0.1, 1.0, 10.0, 100.0, 1000.0]
validation = design.loc[first_validation:first_test - 1].dropna()
if validation.empty:
    raise ValueError("No complete validation rows are available before the test.")

validation_errors = {}
for penalty in penalties:
    predictions = [fit_forecast(design, month, Ridge(alpha=penalty))[0]
                   for month in validation.index]
    squared_errors = validation[target_column].sub(predictions).pow(2)
    validation_errors[penalty] = squared_errors.mean()
tuning = pd.Series(validation_errors, name="Validation MSFE").rename_axis("Ridge penalty")
ridge_penalty = float(tuning.idxmin())
display(tuning.to_frame().round(4))
print(f"Fixed test-period penalty: {ridge_penalty:g} "
      f"({len(validation)} validation targets, {validation.index[0]} to {validation.index[-1]})")

## Make the expanding-window forecasts

The test starts with the January 2010 target. For each target month, train on complete regression rows with outcomes observed by the previous month-end. As more outcomes arrive, use them in the next fit. The historical-mean rule averages the available market returns from January 1990 through the forecast origin. It requires only past market returns, so a month with a missing factor input can still contribute to that mean. Compare every method on the same test dates with observed outcomes and complete inputs.

In [ ]:
forecast_columns = ["Historical mean", "OLS", "Ridge"]
available_inputs = design.loc[first_test:last_test].dropna(subset=feature_columns)
rows = []
for month in available_inputs.index:
    origin = month - 1
    ols, n_train = fit_forecast(design, month, LinearRegression())
    ridge, _ = fit_forecast(design, month, Ridge(alpha=ridge_penalty))
    historical_mean = monthly.loc[first_target:origin, "MKT"].mean()
    rows.append({
        "Target month": month,
        "Forecast origin": origin,
        "Training rows": n_train,
        "Historical mean": historical_mean,
        "OLS": ols,
        "Ridge": ridge,
    })

if not rows:
    raise ValueError("No test forecasts can be formed from these predictor rows.")
forecasts = pd.DataFrame(rows).set_index("Target month")
comparison = forecasts.join(design[target_column]).dropna(subset=[target_column])
if comparison.empty or not np.isfinite(comparison[[target_column, *forecast_columns]]).all().all():
    raise ValueError("Check that all scored dates have an outcome and finite forecasts.")

print(f"Matched test targets: {comparison.index[0]} to {comparison.index[-1]} "
      f"({len(comparison)} of {len(test_calendar)} calendar months)")
print(f"Months without inputs: {len(test_calendar) - len(forecasts)}. "
      f"Forecast months without outcomes: {len(forecasts) - len(comparison)}.")
display(comparison.iloc[[0, 1, -1]].round(3))

## Score the forecasts and check the two periods

A forecast error is the realised return minus the forecast, in percentage points. The mean squared forecast error (MSFE) averages the squared errors, and its square root (RMSFE) returns to percentage points. Report each model's RMSFE and its score relative to the historical-mean rule. The score is $1 - \mathrm{MSFE}_{\mathrm{model}}/\mathrm{MSFE}_{\mathrm{mean}}$ on the same dates. A positive score means smaller squared errors than the benchmark on average.

Keep January 2010 to December 2023 as the main test period. Use 2010–2016 and 2017–2023 as the pre-specified robustness check. The models follow the same procedure in both subperiods. Does the ranking persist, and would the size of the change matter to your decision?

In [ ]:
def loss_summary(sample):
    if sample.empty:
        raise ValueError("There are no matched test months in this period.")
    squared_errors = sample[forecast_columns].sub(sample[target_column], axis=0).pow(2)
    msfe = squared_errors.mean()
    if msfe["Historical mean"] <= 0:
        raise ValueError("The benchmark needs positive MSFE for a relative score.")
    return pd.DataFrame({
        "Months": len(sample),
        "MSFE (squared pp)": msfe,
        "RMSFE (pp)": msfe.pow(0.5),
        "Score vs historical mean": 1 - msfe / msfe["Historical mean"],
    })


periods = {
    "Full test 2010–2023": comparison,
    "2010–2016": comparison.loc[:robustness_break - 1],
    "2017–2023": comparison.loc[robustness_break:],
}
results = pd.concat({name: loss_summary(sample) for name, sample in periods.items()},
                    names=["Target period", "Forecast"])
display(results.round(4))

squared_errors = comparison[forecast_columns].sub(comparison[target_column], axis=0).pow(2)
extra_loss = squared_errors[["OLS", "Ridge"]].sub(squared_errors["Historical mean"], axis=0)
cumulative = extra_loss.cumsum()
cumulative.index = cumulative.index.to_timestamp()
ax = cumulative.plot(figsize=(8, 4), linewidth=1.8)
ax.axhline(0, color="0.4", linewidth=1)
ax.axvline(robustness_break.to_timestamp(), color="0.5", linestyle="--", linewidth=1)
ax.set(title=f"Cumulative squared error above the historical mean | {market_names[market]}",
       xlabel="Target month", ylabel="Extra squared error (squared pp)")
plt.tight_layout()
plt.show()

## Write your recommendation

A rising line shows where a model accumulates more squared error than the historical mean. A falling line shows where it closes that gap or moves ahead. Use the table to check what the two subperiods support. Explain what happens in both comparisons, the move from OLS to ridge and the choice between either fitted model and the historical mean.

State which procedure you would use and what evidence might change your decision. Discuss the magnitude and stability of the forecast errors and one practical constraint, such as data availability or the cost of acting on small forecast changes. If you propose changing an allocation, explain what else you would check before committing money. Record any variants you try after you have seen the test results.